### Sistema Recomendación Híbrido

El objetivo de este proyecto es construir un sistema de recomendación basado en las compras realizadas por los clientes de una superficie comercial. El enfoque parte de una restricción clave: al tratarse de compra física, no es posible recomendar en tiempo real durante la compra, ya que no conocemos la cesta del cliente hasta que pasa por caja. Esto diferencia el problema del de una tienda online, donde sí se puede recomendar según la cesta se va componiendo.
Por ello, el sistema se orienta a la recomendación entre visitas: a partir del historial de compra de cada cliente, generar recomendaciones personalizadas que incentiven su siguiente visita (mediante email, app o cupones en el ticket), en línea con los programas de fidelización habituales en el sector.
Para ello se construyen dos modelos:

- ALS, que aprende los hábitos de cada cliente a partir de su historial para predecir y recomendar su próxima compra. Es el núcleo de la personalización.
- Popularidad, que recomienda los productos más vendidos como red de seguridad para clientes nuevos o sin historial suficiente.

### Librerías

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [ ]:
import pandas as pd
import numpy as np
# ALS
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
# Optimización
import optuna

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

In [51]:
df = pd.read_pickle('data/transacciones_limpio.pkl')
print('Datos cargados correctamente')
df.head(5)

Datos cargados correctamente


,line_id,ticket_number,user_card_id,payment_method,shop_name,shop_address,product_code,product_name,seccion,units,price_per_unit,price_total,created_at
0,9526401,6567014,101629,TARJETA,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1063,HELADO CHOCOLATE 1L,Congelados,1,3.38,3.38,2025-01-01 08:20:00
1,9526398,6567014,101629,TARJETA,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1122,ACEITUNAS RELLENAS 350G,Snacks,3,1.77,5.31,2025-01-01 08:20:00
2,9526399,6567014,101629,TARJETA,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1123,PALOMITAS MICROONDAS PACK 3,Snacks,1,2.27,2.27,2025-01-01 08:20:00
3,9526400,6567014,101629,TARJETA,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1124,NACHOS 200G,Snacks,1,1.96,1.96,2025-01-01 08:20:00
4,9526403,6567014,101629,TARJETA,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1070,CROQUETAS JAMON 500G,Precocinados,2,3.17,6.34,2025-01-01 08:20:00


Comprobamos longitud datos y fecha mínima y máxima para realizar división de datos

In [4]:
print(f'Filas: {len(df):,}')
print(df.created_at.min())
print(df.created_at.max())

Filas: 100,000
2025-01-01 08:20:00
2025-06-30 21:58:00


Tras comprobar fechas procedemos a separar los datos y crear un conjunto de entrenamiento y testeo. Esta separación se realiza de forma manual.

In [5]:
# Fecha corte. Últimos 30 días para test
fecha_corte = df.created_at.max() - pd.Timedelta(days=30)
print(f'Fecha corte: {fecha_corte}')

# creamos conjuntos
train = df[df.created_at <= fecha_corte].copy()
test = df[df.created_at > fecha_corte].copy()

print(f'Longitud Train: {len(train)}; ({len(train)/len(df)*100:.0f}%)') 
print(f'Longitud Test: {len(test)}; ({len(test)/len(df)*100:.0f}%)')
print('-'*50)
print(f'Tickets train: {train.ticket_number.nunique():,}') # comprobación de tickets por conjunto
print(f'Tickets test: {test.ticket_number.nunique():,}')


Fecha corte: 2025-05-31 21:58:00
Longitud Train: 83443; (83%)
Longitud Test: 16557; (17%)
--------------------------------------------------
Tickets train: 7,944
Tickets test: 1,596


Ahora vamos a comprobar el número de clientes evaluables. Este paso es importante porque determina la frontera a partir de la cual entra en juego cada modelo. Para ALS, un sistema personalizado, se necesita un registro histórico del cliente por tanto no funcionará con aquellos clientes nuevos, lo que se conoce como cold start. Estos clientes nuevos, son aquellos que solo figuran en el conjunto de test y por tanto no tienen historial previo.

In [6]:
# almacenamos los clientes en conjuntos y eliminamos duplicados
clientes_train = set(train.user_card_id.unique())
clientes_test = set(test.user_card_id.unique())

evaluables = clientes_test & clientes_train # intersección que guarda solo clientes que se encuentran en ambos conjuntos. Aparecen tanto en train como test
cold_start = clientes_test - clientes_train  # clientes que compraron por primera vez dentro del rango de test.

print(f'Clientes en test: {len(clientes_test)}')
print(f'Evaluables: {len(evaluables)}') # clientes sobre los que se puede probar ALS
print(f'Cold start puro (test): {len(cold_start)}')

Clientes en test: 1083
Evaluables: 998
Cold start puro (test): 85


Con estos cálculos ya tenemos una idea sobre la cantidad de clientes que vamos a manejar en las evaluaciones de modelos.
Contamos con 1083 clientes dentro del conjunto de testeo de los cuales, 85 de ellos han realizado su primera compra en el periodo que contempla el conjunto de test. Los restantes 998 clientes se encuentran en ambos conjuntos y nos ayudarán a desarrollar ALS. 

In [8]:
# Productos comprados por cada cliente en test
compras_test = test.groupby('user_card_id')['product_name'].apply(set)

# Productos comprados por cada cliente en train
compras_train = train.groupby('user_card_id')['product_name'].apply(set)

### Modelo Basado en Popularidad

El primer modelo que se construye es un sistema NO personalizado. Este modelo solo se va a fijar en cuantas veces aparece cada producto en cada ticket. Se trata de un modelo totalmente sencillo que no necesita conocer ningún cliente, es ideal para cuando estamos empezando o el cliente es nuevo, se usa como referencia base.

In [ ]:
# Ranking de productos más populares en conjunto train.
popularidad = train.groupby('product_name')['ticket_number'].nunique().sort_values(ascending=False)

def popularidad_producto(cliente, n=10, filtrar=True):
    """
    Función que recomienda los n productos más populares que el cliente aún no ha comprado en conjunto train.

    Parámetros:
    cliente: ID de cada cliente
    n: número de recomendaciones a devolver
    filtrar: True para excluir productos ya comprados por el cliente en la recomendación.

    Devuelve lista de productos recomendados
    """
    productos_comprados = compras_train.get(cliente, set()) if filtrar else set() # recuperamos productos comprados en train por cada cliente
    recomendaciones = [product for product in popularidad.index if product not in productos_comprados] # se genera la recomendación de productos que no se encuentren dentro de los comprados en train
    return recomendaciones[:n]



Se crea una variable en donde se almacenan los productos comprados en cada ticket, esta variable queda excluida de la función para evitar recalcular en cada llamada a la función a esta variable. 

Dentro de la función, se filtran los productos comprados en el conjunto train. El filtrado dependerá de si la condición filtrar es True (valor por defecto). De esta forma, se evita recomendar productos que el cliente ya ha comprado para así recomendarle nuevos productos.

In [14]:
ejemplo = list(evaluables)[0]
print(popularidad_producto(ejemplo, n=10))

['CARNE PICADA MIXTA 500G', 'TOMATE FRITO 400G', 'CEBOLLA KG', 'PATATA KG', 'PASTA ESPAGUETI 500G', 'GALLETAS MARIA 800G', 'REFRESCO COLA 2L', 'MANZANA GOLDEN KG', 'FRIEGASUELOS 1.5L', 'LECHUGA ICEBERG UD']


In [ ]:
def evaluar_modelo(funcion_recomendar, clientes, k=10, **kwargs):
    """
    Función para evaluar un modelo de recomendación con métricas Precision@k y Recall@k.
    
    Parámetros:
    función_recomendar: función que contiene un sistema recomendador
    clientes: lista de clientes que se van evaluar
    k = nº de recomendaciones a considerar por cliente
    **kwargs: argumentos adicionales incluidos en funcion_recomendar.

    Devuelve los valores medios de Precision y Recall.
    """
    # iniciamos listas para guardar punutación de cada cliente
    precisiones = [] 
    recalls = []

    # bucle para recorrer cada cliente
    for cliente in clientes:
        recomendados = set(funcion_recomendar(cliente, n=k, **kwargs))
        comprados = compras_test.get(cliente, set()) # recuperamos productos comprados de conjunto test

        if len(comprados) == 0: # si no hay productos comprados indicamos que siga
            continue

        aciertos = len(recomendados & comprados)
        precisiones.append(aciertos / k)
        recalls.append(aciertos / len(comprados))

    return np.mean(precisiones), np.mean(recalls)

Creamos una función para evaluar los modelos de forma global.

In [35]:
prec_filtro, rec_filtro = evaluar_modelo(popularidad_producto, evaluables, k=10, filtrar=True)
prec_sin, rec_sin = evaluar_modelo(popularidad_producto, evaluables, k=10, filtrar=False)

print(f'Popularidad con filtro: Precision@10= {prec_filtro:.4f} Recall@10={rec_filtro:.4f}')
print(f'Popularidad sin filtro: Precision@10= {prec_sin:.4f} Recall@10={rec_sin:.4f}')

Popularidad con filtro: Precision@10= 0.1249 Recall@10=0.0912
Popularidad sin filtro: Precision@10= 0.4695 Recall@10=0.3712


In [23]:
# # Recomendaciones del baseline para varios clientes evaluables
# for cliente in list(evaluables)[:5]:
#     recomendados = popularidad_producto(cliente, n=5, filtrar=True)
#     ya_compra = compras_train.get(cliente, set())
    
#     print(f"Cliente {cliente}")
#     print(f"  Ya compraba (train): {list(ya_compra)[:6]}")
#     print(f"  Le recomendamos:     {recomendados}")
#     print()

### ALS

Con el sistema basado en la popularidad de los productos como baseline desarrollado, pasamos al sistema más complejo, ALS, en donde se busca realizar recomendaciones personalizadas a cada cliente en base a sus gustos personales.

La idea central de este algoritmo es la factorización de matrices. Para ello partimos de una matriz cliente-producto, que en general suele ser una tabla de gran tamaño que se encuentra principalmente vacía, ya que cada celda muestra el número de veces que el cliente compró cada producto. En un entorno real, es lógico que cada cliente va a comprar una serie de productos específicos y no todo el catálogo de productos, de ahí que se encuentre principalmente vacía.

Una vez tenemos esta tabla/matriz, ALS busca descomponer la matriz en dos matrices más pequeñas, una de clientes y otra de productos, descritas por factores latentes. Los factores latentes son características ocultas que el modelo inventa para explicar los datos. Similar al ajuste de parámetros en un modelo de ML clásico.
Con las dos matrices ya separadas, ALS procede a fijar una de ellas para calcular la otra. Una vez resuelve una de ellas procede a bloquear la otra matriz para resolver la que se encuentra pendiente. Con esto se procede a realizar la predicción, que consiste en multiplicar los vectores de factores.

En primer lugar, vamos a construir la matriz dispersa de cliente-producto

In [29]:
def construir_matriz(train):
    """
    Función que construye matriz dispera a partir de la frecuencia de compra del conjunto train.
    """

    # frecuencia de compra
    interacciones = (
        train.groupby(['user_card_id', 'product_name'])
        .size()
        .reset_index(name='frecuencia')
    )

    # lista de valores únicos para clientes y productos
    clientes_unicos = interacciones['user_card_id'].unique()
    productos_unicos = interacciones['product_name'].unique()

    # mapeo de valores
    cliente_idx = {c: i for i, c in enumerate(clientes_unicos)} #asignaos valor único a cada cliente
    producto_idx = {p: i for i, p in enumerate(productos_unicos)} # asignamos valor único a cada producto
    idx_producto = {i: p for p, i in producto_idx.items()}

    # matriz dispersa
    filas = interacciones['user_card_id'].map(cliente_idx)
    columnas = interacciones['product_name'].map(producto_idx)
    valores = interacciones['frecuencia'].astype(float)

    matriz = csr_matrix(
        (valores, (filas,columnas)),
        shape=(len(clientes_unicos), len(productos_unicos))
    )

    return matriz, cliente_idx, producto_idx, idx_producto 




In [ ]:
matriz_cp, cliente_idx, producto_idx, idx_producto = construir_matriz(train)

print(f'Forma: {matriz_cp.shape}')
print(f'Densidad: {matriz_cp.nnz / (matriz_cp.shape[0]*matriz_cp.shape[1])*100:.2f}%')

Forma: (2197, 138)
Densidad: 15.17%


Una vez se ha construido la matriz, vamos a entrenar un modelo base que luego se optimiza mediante Optuna.

In [31]:
modelo_als_base = AlternatingLeastSquares(random_state=42)
modelo_als_base.fit(matriz_cp)

print('Modelo base entrenado')
print(f'Factores por defecto: {modelo_als_base.factors}')

  0%|          | 0/15 [00:00<?, ?it/s]

Modelo base entrenado
Factores por defecto: 100


Con el modelo base instanciado, vamos a crear una función similar a popularidad_producto pero enfocada en ALS. De esta forma evitamos repetir código de forma innecesaria

In [32]:
def recomendar_als(cliente, n=10, filtrar=True, modelo=None):
    """
    Función que recomienda n productos para un cliente mediante ALS

    Parámetros
    cliente: cliente a evaluar
    n: nº de recomendaciones
    filtrar: True por defecto para no recomendar productos ya comprados
    modelo: None por defecto. Se indica manualmente que modelo de ALS se quiere emplear.
    """

    c_idx = cliente_idx[cliente]
    ids, _ = modelo.recommend(
        c_idx,
        matriz_cp[c_idx],
        N=n,
        filter_already_liked_items=filtrar
    )

    return [idx_producto[i] for i in ids]

In [34]:
prec, rec = evaluar_modelo(recomendar_als, evaluables, k=10, filtrar=True, modelo=modelo_als_base)
print(f"ALS base (con filtro): Precision@10={prec:.4f}  Recall@10={rec:.4f}")

ALS base (con filtro): Precision@10=0.0722  Recall@10=0.0574


Como era de esperar, un modelo base sin optimizar devuelve unos resultados pobres. Esto no quiere decir que una vez se optimice el modelo los resultados vayan a ser mejores. La cuestión es que para un catálogo de 168 productos, 100 factores latentes es una cifra muy grande.

Con esto en mente vamos a proceder con la optimmización del modelo.

In [46]:
def objetivo(trial):
    # creamos espacio de búsqueda
    factors = trial.suggest_int('factors', 2, 30)
    regularization = trial.suggest_float('regularization', 0.001, 1.0, log=True)
    alpha = trial.suggest_float('alpha', 1.0, 40.0)
    iterations = trial.suggest_int('iterations', 10, 40)

    modelo = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        alpha=alpha,
        iterations=iterations,
        random_state=42
    )
    modelo.fit(matriz_cp)

    prec, _ = evaluar_modelo(recomendar_als, evaluables, k=10, filtrar=True, modelo=modelo)
    return prec

In [47]:
optuna.logging.set_verbosity(optuna.logging.WARNING)  # menos ruido en la salida

study = optuna.create_study(direction="maximize")
study.optimize(objetivo, n_trials=30, show_progress_bar=True)

print(f'Mejor Precision@10: {study.best_value:.4f}')
print(f'Mejores parámetros: {study.best_params}')

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/34 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

...

  0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/33 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

Mejor Precision@10: 0.2435
Mejores parámetros: {'factors': 7, 'regularization': 0.0016776716138101678, 'alpha': 3.9937719152959517, 'iterations': 25}


In [48]:
modelo_als_optimizado = AlternatingLeastSquares(
    **study.best_params,
    random_state=42
)
modelo_als_optimizado.fit(matriz_cp)

prec, rec = evaluar_modelo(recomendar_als, evaluables, k=10, filtrar=True, modelo=modelo_als_optimizado)
print(f"ALS final: Precision@10={prec:.4f}  Recall@10={rec:.4f}")

  0%|          | 0/25 [00:00<?, ?it/s]

ALS final: Precision@10=0.2435  Recall@10=0.1817


In [50]:
for cliente in list(evaluables)[:6]:
    recomendados = recomendar_als(cliente, n=5, filtrar=True, modelo=modelo_als_optimizado)
    ya_compra = compras_train.get(cliente, set())
    
    print(f"Cliente {cliente}")
    print(f"  Compra habitual: {list(ya_compra)[:6]}")
    print(f"  Le recomendamos: {recomendados}")
    print()

Cliente 100352
  Compra habitual: ['TE VERDE 25 BOLSAS', 'LECHE ENTERA BRIK 1L', 'MERMELADA FRESA 350G', 'TOMATE RAMA KG', 'AZUCAR BLANCO 1KG', 'CAFE MOLIDO 250G']
  Le recomendamos: ['GALLETAS MARIA 800G', 'TOMATE FRITO 400G', 'CARNE PICADA MIXTA 500G', 'HARINA TRIGO 1KG', 'PASTA ESPAGUETI 500G']

Cliente 102402
  Compra habitual: ['POLLO ASADO ENTERO', 'PATATAS FRITAS BOLSA 150G', 'ENSALADA CESAR PREPARADA', 'FRUTOS SECOS MIX 200G', 'CERVEZA ARTESANA 33CL', 'PATATAS FRITAS CONGELADAS 1KG']
  Le recomendamos: ['HELADO CHOCOLATE 1L', 'PIZZA CONGELADA 4 ESTACIONES', 'EMPANADA ATUN', 'ACEITUNAS RELLENAS 350G', 'CERVEZA PACK 6']

Cliente 102403
  Compra habitual: ['PASTA ESPAGUETI 500G', 'LECHE ENTERA BRIK 1L', 'PIMIENTO ROJO KG', 'ZANAHORIA KG', 'MIEL PURA 500G', 'AZUCAR BLANCO 1KG']
  Le recomendamos: ['PLATANO KG', 'GAMBA FRESCA KG', 'FILETE TERNERA KG', 'MERLUZA FILETE KG', 'CEREALES DESAYUNO 500G']

Cliente 102404
  Compra habitual: ['PALOMITAS MICROONDAS PACK 3', 'TONICA PACK 4', 'V